In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
tf.random.set_seed(1234)
np.random.seed(1234)

# Projectile motion parameters
g = 9.81  # Acceleration due to gravity (m/s^2)
v0 = 20   # Initial velocity (m/s)
theta = np.pi / 4  # Launch angle (45 degrees)

# PINN model
class ProjectileMotionPINN(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.hidden1 = tf.keras.layers.Dense(20, activation='tanh')
        self.hidden2 = tf.keras.layers.Dense(20, activation='tanh')
        self.output_layer = tf.keras.layers.Dense(2)  # x and y coordinates

    def call(self, inputs):
        x = self.hidden1(inputs)
        x = self.hidden2(x)
        return self.output_layer(x)

# Create the model
model = ProjectileMotionPINN()


In [ ]:

# Loss function
def pinn_loss(t):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(t)
        predictions = model(t)
        x, y = predictions[:, 0], predictions[:, 1]
        
        dx_dt = tape.gradient(x, t)
        dy_dt = tape.gradient(y, t)
        
    d2x_dt2 = tape.gradient(dx_dt, t)
    d2y_dt2 = tape.gradient(dy_dt, t)
    
    # Physics-informed loss
    loss_x = d2x_dt2  # No acceleration in x-direction
    loss_y = d2y_dt2 + g  # Acceleration in y-direction due to gravity
    
    # Initial conditions
    ic_x = x[0] - 0  # x(0) = 0
    ic_y = y[0] - 0  # y(0) = 0
    ic_dx = dx_dt[0] - v0 * tf.cos(theta)  # dx/dt(0) = v0 * cos(theta)
    ic_dy = dy_dt[0] - v0 * tf.sin(theta)  # dy/dt(0) = v0 * sin(theta)
    
    # Compute total loss
    mse = tf.reduce_mean(tf.square(loss_x) + tf.square(loss_y) + 
                         tf.square(ic_x) + tf.square(ic_y) + 
                         tf.square(ic_dx) + tf.square(ic_dy))
    
    return mse


In [ ]:

# Training loop
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

# The train_step function encapsulates everything needed for one step of training:
# Forward pass (compute predictions and loss).
# Backward pass (compute gradients).
# Update step (apply gradients to update model parameters).

@tf.function
def train_step(t):
    with tf.GradientTape() as tape:
        loss = pinn_loss(t)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss



In [ ]:

model = ProjectileMotionPINN()



# Generate training data
num_samples = 1000
t_train = tf.random.uniform((num_samples, 1), minval=0, maxval=5)

# Evaluation data
t_eval = np.linspace(0, 5, 100).reshape(-1, 1)

# Training
epochs = 10000
history = []
for epoch in range(epochs):
    loss = train_step(t_train)
    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.numpy():.6f}")
        predictions = model(t_eval).numpy()
        history.append((epoch, predictions))

# Analytical solution
t = t_eval.flatten()
x_analytical = v0 * np.cos(theta) * t
y_analytical = v0 * np.sin(theta) * t - 0.5 * g * t**2


In [ ]:

# Plot learning progress
plt.figure(figsize=(12, 8))
for i, (epoch, pred) in enumerate(history):
    x_pred, y_pred = pred[:, 0], pred[:, 1]
    plt.plot(x_pred, y_pred, label=f'Epoch {epoch}', alpha=(i+1)/len(history))

plt.plot(x_analytical, y_analytical, label='Analytical solution', linestyle='--', color='black', linewidth=2)
plt.xlabel('X-position (m)')
plt.ylabel('Y-position (m)')
plt.title('Projectile Motion Trajectory: PINN Learning Progress')
plt.legend(loc='upper right')
plt.grid(True)
plt.axis('equal')
plt.show()

# Final prediction plot
final_predictions = model(t_eval).numpy()
x_pred, y_pred = final_predictions[:, 0], final_predictions[:, 1]

plt.figure(figsize=(10, 6))
plt.plot(x_pred, y_pred, label='Final PINN prediction')
plt.plot(x_analytical, y_analytical, label='Analytical solution', linestyle='--')
plt.xlabel('X-position (m)')
plt.ylabel('Y-position (m)')
plt.title('Projectile Motion Trajectory: Final Prediction vs Analytical Solution')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
torch.manual_seed(1234)
np.random.seed(1234)

# Projectile motion parameters
g = 9.81  # Acceleration due to gravity (m/s^2)
v0 = 20   # Initial velocity (m/s)
theta = np.pi / 4  # Launch angle (45 degrees)


# PINN model in PyTorch
class ProjectileMotionPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden1 = nn.Sequential(nn.Linear(1, 20), nn.Tanh())
        self.hidden2 = nn.Sequential(nn.Linear(20, 20), nn.Tanh())
        self.output_layer = nn.Linear(20, 2)  # x and y coordinates

    def forward(self, t):
        x = self.hidden1(t)
        x = self.hidden2(x)
        return self.output_layer(x)


# Instantiate model
model = ProjectileMotionPINN()

# Loss function
def pinn_loss(t):
    t.requires_grad_(True)
    pred = model(t)
    x = pred[:, 0]
    y = pred[:, 1]

    dx_dt = torch.autograd.grad(x, t, grad_outputs=torch.ones_like(x), create_graph=True)[0]
    dy_dt = torch.autograd.grad(y, t, grad_outputs=torch.ones_like(y), create_graph=True)[0]

    d2x_dt2 = torch.autograd.grad(dx_dt, t, grad_outputs=torch.ones_like(dx_dt), create_graph=True)[0]
    d2y_dt2 = torch.autograd.grad(dy_dt, t, grad_outputs=torch.ones_like(dy_dt), create_graph=True)[0]

    # Physics-informed loss
    loss_x = d2x_dt2  # Should be zero
    loss_y = d2y_dt2 + g  # Should match gravity

    # Initial conditions (at t=0)
    x0 = x[0]
    y0 = y[0]
    dx0 = dx_dt[0]
    dy0 = dy_dt[0]

    ic_x = x0 - 0
    ic_y = y0 - 0
    ic_dx = dx0 - v0 * np.cos(theta)
    ic_dy = dy0 - v0 * np.sin(theta)

    # Mean squared error
    mse = (loss_x**2).mean() + (loss_y**2).mean() + \
          ic_x**2 + ic_y**2 + ic_dx**2 + ic_dy**2

    return mse


# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training data
num_samples = 1000
t_train = torch.FloatTensor(num_samples, 1).uniform_(0, 5)

# Evaluation data
t_eval = np.linspace(0, 5, 100).reshape(-1, 1)
t_eval_tensor = torch.tensor(t_eval, dtype=torch.float32)

# Training loop
epochs = 10000
history = []

for epoch in range(epochs):
    optimizer.zero_grad()
    loss = pinn_loss(t_train)
    loss.backward()
    optimizer.step()
    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
        with torch.no_grad():
            predictions = model(t_eval_tensor).numpy()
            history.append((epoch, predictions))

# Analytical solution
t = t_eval.flatten()
x_analytical = v0 * np.cos(theta) * t
y_analytical = v0 * np.sin(theta) * t - 0.5 * g * t**2

# Optional: Plot predictions vs analytical
with torch.no_grad():
    pred = model(t_eval_tensor).numpy()

plt.figure()
plt.plot(t, x_analytical, label='x (analytical)', linestyle='--')
plt.plot(t, y_analytical, label='y (analytical)', linestyle='--')
plt.plot(t, pred[:, 0], label='x (PINN)')
plt.plot(t, pred[:, 1], label='y (PINN)')
plt.xlabel('Time (s)')
plt.ylabel('Position (m)')
plt.legend()
plt.title("Projectile Motion: PINN vs Analytical")
plt.grid()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Plot learning progress
plt.figure(figsize=(12, 8))
for i, (epoch, pred) in enumerate(history):
    x_pred, y_pred = pred[:, 0], pred[:, 1]
    plt.plot(x_pred, y_pred, label=f'Epoch {epoch}', alpha=(i + 1) / len(history))

plt.plot(x_analytical, y_analytical, label='Analytical solution', linestyle='--', color='black', linewidth=2)
plt.xlabel('X-position (m)')
plt.ylabel('Y-position (m)')
plt.title('Projectile Motion Trajectory: PINN Learning Progress')
plt.legend(loc='upper right')
plt.grid(True)
plt.axis('equal')  # Keep x and y scales the same
plt.show()


In [ ]:
# Final prediction
with torch.no_grad():
    final_predictions = model(t_eval_tensor).numpy()

x_pred, y_pred = final_predictions[:, 0], final_predictions[:, 1]

plt.figure(figsize=(10, 6))
plt.plot(x_pred, y_pred, label='Final PINN prediction', linewidth=2)
plt.plot(x_analytical, y_analytical, label='Analytical solution', linestyle='--', linewidth=2)
plt.xlabel('X-position (m)')
plt.ylabel('Y-position (m)')
plt.title('Projectile Motion Trajectory: Final Prediction vs Analytical Solution')
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.show()


In [ ]:
"""
Physics-Informed Neural Networks (PINNs) for Navier-Stokes Equations
===================================================================

This lab demonstrates how to implement and train a PINN to solve the 2D Navier-Stokes equations
for incompressible fluid flow. Students will learn how to:

1. Set up the neural network architecture
2. Implement physics-informed loss functions
3. Handle boundary conditions
4. Train the model and visualize results

The Navier-Stokes equations for incompressible flow are:
- Continuity: ∂u/∂x + ∂v/∂y = 0
- Momentum (x): ∂u/∂t + u∂u/∂x + v∂u/∂y = -∂p/∂x + ν∇²u
- Momentum (y): ∂v/∂t + u∂v/∂x + v∂v/∂y = -∂p/∂y + ν∇²v

Where u, v are velocity components, p is pressure, ν is kinematic viscosity
"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import time

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class NavierStokesPINN(nn.Module):
    """
    Physics-Informed Neural Network for Navier-Stokes equations
    
    The network takes (x, y, t) as input and outputs (u, v, p)
    where u, v are velocity components and p is pressure
    """
    
    def __init__(self, hidden_layers=8, hidden_units=50):
        super(NavierStokesPINN, self).__init__()
        
        # Input layer: (x, y, t) -> 3 inputs
        layers = [nn.Linear(3, hidden_units), nn.Tanh()]
        
        # Hidden layers
        for _ in range(hidden_layers):
            layers.extend([nn.Linear(hidden_units, hidden_units), nn.Tanh()])
        
        # Output layer: -> (u, v, p) -> 3 outputs
        layers.append(nn.Linear(hidden_units, 3))
        
        self.network = nn.Sequential(*layers)
        
        # Initialize weights using Xavier initialization
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, x, y, t):
        """
        Forward pass through the network
        
        Args:
            x, y, t: spatial and temporal coordinates
            
        Returns:
            u, v, p: velocity components and pressure
        """
        inputs = torch.cat([x, y, t], dim=1)
        output = self.network(inputs)
        u, v, p = output[:, 0:1], output[:, 1:2], output[:, 2:3]
        return u, v, p


In [ ]:
class NavierStokesLoss:
    """
    Loss function for Navier-Stokes PINN
    
    Combines:
    1. PDE residuals (physics loss)
    2. Boundary condition losses
    3. Initial condition losses
    """
    
    def __init__(self, nu=0.01):
        """
        Args:
            nu: kinematic viscosity
        """
        self.nu = nu
    
    def pde_residual(self, model, x, y, t):
        """
        Compute PDE residuals for Navier-Stokes equations
        
        Returns:
            continuity_residual: ∂u/∂x + ∂v/∂y
            momentum_x_residual: ∂u/∂t + u∂u/∂x + v∂u/∂y + ∂p/∂x - ν∇²u
            momentum_y_residual: ∂v/∂t + u∂v/∂x + v∂v/∂y + ∂p/∂y - ν∇²v
        """
        # Enable gradient computation
        x.requires_grad_(True)
        y.requires_grad_(True)
        t.requires_grad_(True)
        
        # Forward pass
        u, v, p = model(x, y, t)
        
        # First derivatives
        u_x = torch.autograd.grad(u.sum(), x, create_graph=True)[0]
        u_y = torch.autograd.grad(u.sum(), y, create_graph=True)[0]
        u_t = torch.autograd.grad(u.sum(), t, create_graph=True)[0]
        
        v_x = torch.autograd.grad(v.sum(), x, create_graph=True)[0]
        v_y = torch.autograd.grad(v.sum(), y, create_graph=True)[0]
        v_t = torch.autograd.grad(v.sum(), t, create_graph=True)[0]
        
        p_x = torch.autograd.grad(p.sum(), x, create_graph=True)[0]
        p_y = torch.autograd.grad(p.sum(), y, create_graph=True)[0]
        
        # Second derivatives for Laplacian
        u_xx = torch.autograd.grad(u_x.sum(), x, create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y.sum(), y, create_graph=True)[0]
        
        v_xx = torch.autograd.grad(v_x.sum(), x, create_graph=True)[0]
        v_yy = torch.autograd.grad(v_y.sum(), y, create_graph=True)[0]
        
        # Continuity equation: ∂u/∂x + ∂v/∂y = 0
        continuity = u_x + v_y
        
        # Momentum equations
        momentum_x = u_t + u * u_x + v * u_y + p_x - self.nu * (u_xx + u_yy)
        momentum_y = v_t + u * v_x + v * v_y + p_y - self.nu * (v_xx + v_yy)
        
        return continuity, momentum_x, momentum_y
    
    def boundary_loss(self, model, x_bc, y_bc, t_bc, u_bc, v_bc):
        """
        Compute boundary condition loss
        
        Args:
            x_bc, y_bc, t_bc: boundary coordinates
            u_bc, v_bc: boundary values for velocity
        """
        u_pred, v_pred, _ = model(x_bc, y_bc, t_bc)
        loss_u = torch.mean((u_pred - u_bc)**2)
        loss_v = torch.mean((v_pred - v_bc)**2)
        return loss_u + loss_v
    
    def initial_loss(self, model, x_ic, y_ic, t_ic, u_ic, v_ic):
        """
        Compute initial condition loss
        """
        u_pred, v_pred, _ = model(x_ic, y_ic, t_ic)
        loss_u = torch.mean((u_pred - u_ic)**2)
        loss_v = torch.mean((v_pred - v_ic)**2)
        return loss_u + loss_v


In [ ]:

def generate_training_data(n_pde=2000, n_bc=200, n_ic=200):
    """
    Generate training data for the PINN
    
    Returns:
        Dictionary containing collocation points and boundary/initial conditions
    """
    
    # Domain: [0,1] x [0,1] x [0,1]
    x_min, x_max = 0.0, 1.0
    y_min, y_max = 0.0, 1.0
    t_min, t_max = 0.0, 1.0
    
    # PDE collocation points (interior)
    x_pde = torch.rand(n_pde, 1) * (x_max - x_min) + x_min
    y_pde = torch.rand(n_pde, 1) * (y_max - y_min) + y_min
    t_pde = torch.rand(n_pde, 1) * (t_max - t_min) + t_min
    
    # Boundary conditions (Lid-driven cavity)
    # Top boundary (y=1): u=1, v=0 (moving lid)
    x_bc_top = torch.rand(n_bc//4, 1) * (x_max - x_min) + x_min
    y_bc_top = torch.ones(n_bc//4, 1) * y_max
    t_bc_top = torch.rand(n_bc//4, 1) * (t_max - t_min) + t_min
    u_bc_top = torch.ones(n_bc//4, 1)
    v_bc_top = torch.zeros(n_bc//4, 1)
    
    # Bottom boundary (y=0): u=0, v=0
    x_bc_bottom = torch.rand(n_bc//4, 1) * (x_max - x_min) + x_min
    y_bc_bottom = torch.zeros(n_bc//4, 1)
    t_bc_bottom = torch.rand(n_bc//4, 1) * (t_max - t_min) + t_min
    u_bc_bottom = torch.zeros(n_bc//4, 1)
    v_bc_bottom = torch.zeros(n_bc//4, 1)
    
    # Left boundary (x=0): u=0, v=0
    x_bc_left = torch.zeros(n_bc//4, 1)
    y_bc_left = torch.rand(n_bc//4, 1) * (y_max - y_min) + y_min
    t_bc_left = torch.rand(n_bc//4, 1) * (t_max - t_min) + t_min
    u_bc_left = torch.zeros(n_bc//4, 1)
    v_bc_left = torch.zeros(n_bc//4, 1)
    
    # Right boundary (x=1): u=0, v=0
    x_bc_right = torch.ones(n_bc//4, 1) * x_max
    y_bc_right = torch.rand(n_bc//4, 1) * (y_max - y_min) + y_min
    t_bc_right = torch.rand(n_bc//4, 1) * (t_max - t_min) + t_min
    u_bc_right = torch.zeros(n_bc//4, 1)
    v_bc_right = torch.zeros(n_bc//4, 1)
    
    # Combine boundary conditions
    x_bc = torch.cat([x_bc_top, x_bc_bottom, x_bc_left, x_bc_right])
    y_bc = torch.cat([y_bc_top, y_bc_bottom, y_bc_left, y_bc_right])
    t_bc = torch.cat([t_bc_top, t_bc_bottom, t_bc_left, t_bc_right])
    u_bc = torch.cat([u_bc_top, u_bc_bottom, u_bc_left, u_bc_right])
    v_bc = torch.cat([v_bc_top, v_bc_bottom, v_bc_left, v_bc_right])
    
    # Initial conditions (t=0): u=0, v=0 everywhere
    x_ic = torch.rand(n_ic, 1) * (x_max - x_min) + x_min
    y_ic = torch.rand(n_ic, 1) * (y_max - y_min) + y_min
    t_ic = torch.zeros(n_ic, 1)
    u_ic = torch.zeros(n_ic, 1)
    v_ic = torch.zeros(n_ic, 1)
    
    return {
        'x_pde': x_pde.to(device), 'y_pde': y_pde.to(device), 't_pde': t_pde.to(device),
        'x_bc': x_bc.to(device), 'y_bc': y_bc.to(device), 't_bc': t_bc.to(device),
        'u_bc': u_bc.to(device), 'v_bc': v_bc.to(device),
        'x_ic': x_ic.to(device), 'y_ic': y_ic.to(device), 't_ic': t_ic.to(device),
        'u_ic': u_ic.to(device), 'v_ic': v_ic.to(device)
    }


In [ ]:

def train_pinn(model, data, epochs=5000, lr=1e-3, lambda_pde=1.0, lambda_bc=10.0, lambda_ic=10.0):
    """
    Train the PINN model
    
    Args:
        model: PINN model
        data: training data dictionary
        epochs: number of training epochs
        lr: learning rate
        lambda_pde, lambda_bc, lambda_ic: loss weights
    """
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.9)
    
    loss_fn = NavierStokesLoss(nu=0.01)
    
    losses = {'total': [], 'pde': [], 'bc': [], 'ic': []}
    
    print("Starting training...")
    start_time = time.time()
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # PDE loss
        continuity, momentum_x, momentum_y = loss_fn.pde_residual(
            model, data['x_pde'], data['y_pde'], data['t_pde']
        )
        loss_pde = torch.mean(continuity**2) + torch.mean(momentum_x**2) + torch.mean(momentum_y**2)
        
        # Boundary condition loss
        loss_bc = loss_fn.boundary_loss(
            model, data['x_bc'], data['y_bc'], data['t_bc'], data['u_bc'], data['v_bc']
        )
        
        # Initial condition loss
        loss_ic = loss_fn.initial_loss(
            model, data['x_ic'], data['y_ic'], data['t_ic'], data['u_ic'], data['v_ic']
        )
        
        # Total loss
        total_loss = lambda_pde * loss_pde + lambda_bc * loss_bc + lambda_ic * loss_ic
        
        # Backward pass
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        
        # Store losses
        losses['total'].append(total_loss.item())
        losses['pde'].append(loss_pde.item())
        losses['bc'].append(loss_bc.item())
        losses['ic'].append(loss_ic.item())
        
        # Print progress
        if epoch % 500 == 0:
            print(f"Epoch {epoch:5d} | Total Loss: {total_loss.item():.6f} | "
                  f"PDE: {loss_pde.item():.6f} | BC: {loss_bc.item():.6f} | IC: {loss_ic.item():.6f}")
    
    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time:.2f} seconds")
    
    return losses


In [ ]:

def visualize_results(model, t_eval=0.5):
    """
    Visualize the solution at a given time
    """
    model.eval()
    
    # Create grid for visualization - FIXED: Use correct indexing for matplotlib
    x = torch.linspace(0, 1, 50)
    y = torch.linspace(0, 1, 50)
    X, Y = torch.meshgrid(x, y, indexing='xy')  # Changed from 'ij' to 'xy'
    
    x_flat = X.flatten().reshape(-1, 1).to(device)
    y_flat = Y.flatten().reshape(-1, 1).to(device)
    t_flat = torch.ones_like(x_flat) * t_eval
    
    with torch.no_grad():
        u_pred, v_pred, p_pred = model(x_flat, y_flat, t_flat)
    
    u_pred = u_pred.cpu().numpy().reshape(50, 50)
    v_pred = v_pred.cpu().numpy().reshape(50, 50)
    p_pred = p_pred.cpu().numpy().reshape(50, 50)
    
    X_np = X.numpy()
    Y_np = Y.numpy()
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Velocity magnitude
    velocity_mag = np.sqrt(u_pred**2 + v_pred**2)
    im1 = axes[0, 0].contourf(X_np, Y_np, velocity_mag, levels=20, cmap='viridis')
    axes[0, 0].set_title(f'Velocity Magnitude at t={t_eval}')
    axes[0, 0].set_xlabel('x')
    axes[0, 0].set_ylabel('y')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # Streamlines - FIXED: Now works with correct meshgrid
    axes[0, 1].streamplot(X_np, Y_np, u_pred, v_pred, density=1.5, color='black', linewidth=1)
    axes[0, 1].set_title(f'Streamlines at t={t_eval}')
    axes[0, 1].set_xlabel('x')
    axes[0, 1].set_ylabel('y')
    
    # Pressure
    im3 = axes[1, 0].contourf(X_np, Y_np, p_pred, levels=20, cmap='RdBu')
    axes[1, 0].set_title(f'Pressure at t={t_eval}')
    axes[1, 0].set_xlabel('x')
    axes[1, 0].set_ylabel('y')
    plt.colorbar(im3, ax=axes[1, 0])
    
    # Vorticity - FIXED: Use proper spacing for gradient calculation
    dx = 1.0 / 49  # Grid spacing
    dy = 1.0 / 49
    vorticity = np.gradient(v_pred, dx, axis=1) - np.gradient(u_pred, dy, axis=0)
    im4 = axes[1, 1].contourf(X_np, Y_np, vorticity, levels=20, cmap='RdBu')
    axes[1, 1].set_title(f'Vorticity at t={t_eval}')
    axes[1, 1].set_xlabel('x')
    axes[1, 1].set_ylabel('y')
    plt.colorbar(im4, ax=axes[1, 1])
    
    plt.tight_layout()
    plt.show()

def plot_loss_history(losses):
    """
    Plot training loss history
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    epochs = range(len(losses['total']))
    
    axes[0, 0].semilogy(epochs, losses['total'])
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True)
    
    axes[0, 1].semilogy(epochs, losses['pde'])
    axes[0, 1].set_title('PDE Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].grid(True)
    
    axes[1, 0].semilogy(epochs, losses['bc'])
    axes[1, 0].set_title('Boundary Condition Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].grid(True)
    
    axes[1, 1].semilogy(epochs, losses['ic'])
    axes[1, 1].set_title('Initial Condition Loss')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("PHYSICS-INFORMED NEURAL NETWORKS FOR NAVIER-STOKES")
    print("=" * 60)
    
    # Step 1: Initialize the model
    print("\n1. Initializing PINN model...")
    model = NavierStokesPINN(hidden_layers=8, hidden_units=50).to(device)
    print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
    
    # Step 2: Generate training data
    print("\n2. Generating training data...")
    training_data = generate_training_data(n_pde=2000, n_bc=400, n_ic=200)
    print(f"PDE points: {training_data['x_pde'].shape[0]}")
    print(f"Boundary points: {training_data['x_bc'].shape[0]}")
    print(f"Initial points: {training_data['x_ic'].shape[0]}")
    
    # Step 3: Train the model
    print("\n3. Training the model...")
    losses = train_pinn(model, training_data, epochs=3000, lr=1e-3)
    
    # Step 4: Visualize results
    print("\n4. Visualizing results...")
    plot_loss_history(losses)
    visualize_results(model, t_eval=0.5)
    
    print("\n" + "=" * 60)
    print("LAB COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    
    # Additional exercises for students
    print("\nSUGGESTED EXERCISES:")
    print("1. Modify the Reynolds number (viscosity) and observe the effect")
    print("2. Change the boundary conditions to different flow scenarios")
    print("3. Implement different network architectures and compare performance")
    print("4. Add more collocation points and study convergence")
    print("5. Visualize the solution evolution over time")
    print("6. Compare with analytical solutions for simple cases")

In [ ]:
"""
Physics-Informed Neural Networks for Scientific Discovery and Law Verification
============================================================================

This lab demonstrates how to use PINNs to:
1. Discover governing equations from data
2. Verify known physical laws
3. Identify missing physics in models
4. Quantify model uncertainty

We'll explore several examples:
- Discovering Newton's laws from trajectory data
- Verifying conservation laws
- Finding unknown damping in oscillators
- Discovering reaction kinetics from concentration data
"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import seaborn as sns
from sklearn.metrics import r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class DiscoveryPINN(nn.Module):
    """
    PINN for discovering governing equations from data
    
    Can learn both the solution and the governing equation parameters
    """
    
    def __init__(self, input_dim=1, output_dim=1, hidden_layers=4, hidden_units=50, 
                 learn_params=True, n_params=1):
        super(DiscoveryPINN, self).__init__()
        
        # Neural network for solution
        layers = [nn.Linear(input_dim, hidden_units), nn.Tanh()]
        for _ in range(hidden_layers):
            layers.extend([nn.Linear(hidden_units, hidden_units), nn.Tanh()])
        layers.append(nn.Linear(hidden_units, output_dim))
        
        self.network = nn.Sequential(*layers)
        
        # Learnable physical parameters
        self.learn_params = learn_params
        if learn_params:
            self.physics_params = nn.Parameter(torch.randn(n_params) * 0.1)
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)
    
    def forward(self, x):
        return self.network(x)
    
    def get_physics_params(self):
        if self.learn_params:
            return self.physics_params
        return None


In [ ]:

class NewtonianMechanicsDiscovery:
    """
    Discover Newton's second law from trajectory data
    
    Given position data, learn: F = ma
    """
    
    def __init__(self, true_mass=1.0, true_force_params=None):
        self.true_mass = true_mass
        self.true_force_params = true_force_params or [1.0]  # Force parameters
    
    def generate_data(self, n_points=100, noise_level=0.02):
        """Generate trajectory data with known physics"""
        t = np.linspace(0, 2*np.pi, n_points)
        
        # Example: Harmonic oscillator F = -kx
        k = self.true_force_params[0]
        omega = np.sqrt(k / self.true_mass)
        
        # Analytical solution
        x = np.cos(omega * t)
        v = -omega * np.sin(omega * t)
        a = -omega**2 * np.cos(omega * t)
        
        # Add noise
        x += np.random.normal(0, noise_level, x.shape)
        v += np.random.normal(0, noise_level, v.shape)
        
        return t, x, v, a
    
    def train_discovery_model(self, t, x, epochs=5000, lr=1e-3):
        """Train PINN to discover F=ma relationship"""
        
        # Model learns both position and physical parameters
        model = DiscoveryPINN(input_dim=1, output_dim=1, 
                             learn_params=True, n_params=2).to(device)  # mass and k
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.8)
        
        t_tensor = torch.tensor(t.reshape(-1, 1), dtype=torch.float32).to(device)
        x_tensor = torch.tensor(x.reshape(-1, 1), dtype=torch.float32).to(device)
        
        losses = []
        mass_history = []
        k_history = []
        
        print("Discovering Newton's laws from trajectory data...")
        
        for epoch in range(epochs):
            optimizer.zero_grad()
            
            # Forward pass
            t_tensor.requires_grad_(True)
            x_pred = model(t_tensor)
            
            # Compute derivatives
            x_t = torch.autograd.grad(x_pred.sum(), t_tensor, create_graph=True)[0]
            x_tt = torch.autograd.grad(x_t.sum(), t_tensor, create_graph=True)[0]
            
            # Get learned parameters
            params = model.get_physics_params()
            learned_mass = torch.abs(params[0]) + 0.1  # Ensure positive mass
            learned_k = torch.abs(params[1]) + 0.1     # Ensure positive k
            
            # Physics loss: F = ma, where F = -kx for harmonic oscillator
            # ma = -kx  =>  m * x_tt = -k * x_pred
            physics_loss = torch.mean((learned_mass * x_tt + learned_k * x_pred)**2)
            
            # Data fitting loss
            data_loss = torch.mean((x_pred - x_tensor)**2)
            
            # Total loss
            total_loss = data_loss + 0.1 * physics_loss
            
            total_loss.backward()
            optimizer.step()
            scheduler.step()
            
            losses.append(total_loss.item())
            mass_history.append(learned_mass.item())
            k_history.append(learned_k.item())
            
            if epoch % 1000 == 0:
                print(f"Epoch {epoch:4d} | Loss: {total_loss.item():.6f} | "
                      f"Mass: {learned_mass.item():.3f} | k: {learned_k.item():.3f}")
        
        return model, losses, mass_history, k_history


In [ ]:

class ConservationLawVerification:
    """
    Verify conservation laws (energy, momentum) using PINNs
    """
    
    def __init__(self):
        pass
    
    def pendulum_energy_conservation(self, n_points=200, noise_level=0.01):
        """
        Verify energy conservation in a pendulum system
        """
        # Generate pendulum data
        t = np.linspace(0, 4*np.pi, n_points)
        
        def pendulum_ode(y, t, g=9.81, L=1.0):
            theta, theta_dot = y
            return [theta_dot, -(g/L) * np.sin(theta)]
        
        # Initial conditions
        theta0 = np.pi/4  # 45 degrees
        theta_dot0 = 0.0
        
        # Solve ODE
        solution = odeint(pendulum_ode, [theta0, theta_dot0], t)
        theta = solution[:, 0]
        theta_dot = solution[:, 1]
        
        # Add noise
        theta += np.random.normal(0, noise_level, theta.shape)
        theta_dot += np.random.normal(0, noise_level, theta_dot.shape)
        
        # Calculate theoretical energy
        g, L = 9.81, 1.0
        kinetic_energy = 0.5 * L**2 * theta_dot**2
        potential_energy = g * L * (1 - np.cos(theta))
        total_energy = kinetic_energy + potential_energy
        
        return t, theta, theta_dot, total_energy
    
    def train_conservation_model(self, t, theta, theta_dot, epochs=3000):
        """
        Train PINN to learn pendulum dynamics and verify energy conservation
        """
        model = DiscoveryPINN(input_dim=1, output_dim=2, 
                             learn_params=True, n_params=2).to(device)  # g and L
        
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        
        t_tensor = torch.tensor(t.reshape(-1, 1), dtype=torch.float32).to(device)
        theta_tensor = torch.tensor(theta.reshape(-1, 1), dtype=torch.float32).to(device)
        theta_dot_tensor = torch.tensor(theta_dot.reshape(-1, 1), dtype=torch.float32).to(device)
        
        losses = []
        energy_violations = []
        
        print("Training conservation law verification model...")
        
        for epoch in range(epochs):
            optimizer.zero_grad()
            
            t_tensor.requires_grad_(True)
            output = model(t_tensor)
            theta_pred = output[:, 0:1]
            theta_dot_pred = output[:, 1:2]
            
            # Compute second derivative
            theta_dot_pred_sum = theta_dot_pred.sum()
            theta_dot_t = torch.autograd.grad(theta_dot_pred_sum, t_tensor, create_graph=True)[0]
            
            # Get learned parameters
            params = model.get_physics_params()
            g = torch.abs(params[0]) + 1.0
            L = torch.abs(params[1]) + 0.1
            
            # Physics loss: pendulum equation
            physics_loss = torch.mean((theta_dot_t + (g/L) * torch.sin(theta_pred))**2)
            
            # Data fitting loss
            data_loss = torch.mean((theta_pred - theta_tensor)**2) + \
                       torch.mean((theta_dot_pred - theta_dot_tensor)**2)
            
            # Energy conservation loss
            kinetic = 0.5 * L**2 * theta_dot_pred**2
            potential = g * L * (1 - torch.cos(theta_pred))
            total_energy = kinetic + potential
            
            # Energy should be constant
            energy_loss = torch.var(total_energy)
            
            total_loss = data_loss + 0.1 * physics_loss + 0.01 * energy_loss
            
            total_loss.backward()
            optimizer.step()
            
            losses.append(total_loss.item())
            energy_violations.append(energy_loss.item())
            
            if epoch % 500 == 0:
                print(f"Epoch {epoch:4d} | Loss: {total_loss.item():.6f} | "
                      f"g: {g.item():.2f} | L: {L.item():.3f} | "
                      f"Energy var: {energy_loss.item():.6f}")
        
        return model, losses, energy_violations


In [ ]:

class ReactionKineticsDiscovery:
    """
    Discover chemical reaction kinetics from concentration data
    """
    
    def __init__(self, true_rate_constants=None):
        self.true_rate_constants = true_rate_constants or [0.1, 0.05]
    
    def generate_reaction_data(self, n_points=100, noise_level=0.02):
        """
        Generate data for A + B -> C reaction
        Rate equations: dA/dt = -k1*A*B, dB/dt = -k1*A*B, dC/dt = k1*A*B
        """
        t = np.linspace(0, 20, n_points)
        
        def reaction_ode(y, t, k1):
            A, B, C = y
            dAdt = -k1 * A * B
            dBdt = -k1 * A * B
            dCdt = k1 * A * B
            return [dAdt, dBdt, dCdt]
        
        # Initial concentrations
        y0 = [1.0, 0.8, 0.0]  # A, B, C
        
        # Solve ODE
        solution = odeint(reaction_ode, y0, t, args=(self.true_rate_constants[0],))
        
        # Add noise
        solution += np.random.normal(0, noise_level, solution.shape)
        
        return t, solution
    
    def train_kinetics_model(self, t, concentrations, epochs=4000):
        """
        Discover reaction rate constants from concentration data
        """
        model = DiscoveryPINN(input_dim=1, output_dim=3,
                             learn_params=True, n_params=1).to(device)  # k1
        
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        
        t_tensor = torch.tensor(t.reshape(-1, 1), dtype=torch.float32).to(device)
        conc_tensor = torch.tensor(concentrations, dtype=torch.float32).to(device)
        
        losses = []
        k_history = []
        
        print("Discovering reaction kinetics...")
        
        for epoch in range(epochs):
            optimizer.zero_grad()
            
            t_tensor.requires_grad_(True)
            output = model(t_tensor)
            A_pred, B_pred, C_pred = output[:, 0:1], output[:, 1:2], output[:, 2:3]
            
            # Compute time derivatives
            A_t = torch.autograd.grad(A_pred.sum(), t_tensor, create_graph=True)[0]
            B_t = torch.autograd.grad(B_pred.sum(), t_tensor, create_graph=True)[0]
            C_t = torch.autograd.grad(C_pred.sum(), t_tensor, create_graph=True)[0]
            
            # Get learned rate constant
            k1 = torch.abs(model.get_physics_params()[0]) + 0.001
            
            # Physics loss: reaction rate equations
            rate_AB = k1 * A_pred * B_pred
            physics_loss = torch.mean((A_t + rate_AB)**2) + \
                          torch.mean((B_t + rate_AB)**2) + \
                          torch.mean((C_t - rate_AB)**2)
            
            # Data fitting loss
            data_loss = torch.mean((output - conc_tensor)**2)
            
            # Conservation loss: A0 + B0 + C0 = A + B + C (mass balance)
            total_mass = A_pred + B_pred + C_pred
            initial_mass = conc_tensor[0, 0] + conc_tensor[0, 1] + conc_tensor[0, 2]
            conservation_loss = torch.mean((total_mass - initial_mass)**2)
            
            total_loss = data_loss + 0.1 * physics_loss + 0.01 * conservation_loss
            
            total_loss.backward()
            optimizer.step()
            
            losses.append(total_loss.item())
            k_history.append(k1.item())
            
            if epoch % 800 == 0:
                print(f"Epoch {epoch:4d} | Loss: {total_loss.item():.6f} | k1: {k1.item():.4f}")
        
        return model, losses, k_history


In [ ]:

def visualize_discovery_results():
    """
    Run all discovery experiments and visualize results
    """
    
    # ==========================================
    # 1. NEWTON'S LAW DISCOVERY
    # ==========================================
    print("\n" + "="*60)
    print("1. DISCOVERING NEWTON'S SECOND LAW")
    print("="*60)
    
    newton_exp = NewtonianMechanicsDiscovery(true_mass=2.0, true_force_params=[3.0])
    t, x, v, a = newton_exp.generate_data(n_points=150, noise_level=0.05)
    
    model_newton, losses_newton, mass_hist, k_hist = newton_exp.train_discovery_model(t, x)
    
    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Trajectory comparison
    t_test = np.linspace(0, 2*np.pi, 200)
    t_tensor = torch.tensor(t_test.reshape(-1, 1), dtype=torch.float32).to(device)
    with torch.no_grad():
        x_pred = model_newton(t_tensor).cpu().numpy()
    
    axes[0, 0].plot(t, x, 'ro', alpha=0.6, label='Noisy Data')
    axes[0, 0].plot(t_test, x_pred, 'b-', linewidth=2, label='PINN Prediction')
    axes[0, 0].set_xlabel('Time')
    axes[0, 0].set_ylabel('Position')
    axes[0, 0].set_title('Trajectory Fitting')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Parameter discovery
    epochs = range(len(mass_hist))
    axes[0, 1].plot(epochs, mass_hist, 'r-', label=f'Learned Mass (True: {newton_exp.true_mass})')
    axes[0, 1].plot(epochs, k_hist, 'b-', label=f'Learned k (True: {newton_exp.true_force_params[0]})')
    axes[0, 1].axhline(y=newton_exp.true_mass, color='r', linestyle='--', alpha=0.7)
    axes[0, 1].axhline(y=newton_exp.true_force_params[0], color='b', linestyle='--', alpha=0.7)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Parameter Value')
    axes[0, 1].set_title('Parameter Discovery')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # ==========================================
    # 2. CONSERVATION LAW VERIFICATION
    # ==========================================
    print("\n" + "="*60)
    print("2. VERIFYING ENERGY CONSERVATION")
    print("="*60)
    
    conservation_exp = ConservationLawVerification()
    t_pend, theta, theta_dot, true_energy = conservation_exp.pendulum_energy_conservation()
    
    model_pend, losses_pend, energy_violations = conservation_exp.train_conservation_model(
        t_pend, theta, theta_dot)
    
    # Energy conservation plot
    axes[1, 0].plot(t_pend, true_energy, 'g-', linewidth=2, label='True Energy')
    axes[1, 0].set_xlabel('Time')
    axes[1, 0].set_ylabel('Energy')
    axes[1, 0].set_title('Energy Conservation Verification')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Energy violation over training
    axes[1, 1].semilogy(energy_violations, 'purple', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Energy Variance')
    axes[1, 1].set_title('Energy Conservation Error')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # ==========================================
    # 3. REACTION KINETICS DISCOVERY
    # ==========================================
    print("\n" + "="*60)
    print("3. DISCOVERING REACTION KINETICS")
    print("="*60)
    
    kinetics_exp = ReactionKineticsDiscovery(true_rate_constants=[0.15])
    t_rxn, concentrations = kinetics_exp.generate_reaction_data(noise_level=0.03)
    
    model_rxn, losses_rxn, k_rxn_hist = kinetics_exp.train_kinetics_model(
        t_rxn, concentrations)
    
    # Plot reaction results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Concentration profiles
    t_test = np.linspace(0, 20, 200)
    t_tensor = torch.tensor(t_test.reshape(-1, 1), dtype=torch.float32).to(device)
    with torch.no_grad():
        conc_pred = model_rxn(t_tensor).cpu().numpy()
    
    axes[0, 0].plot(t_rxn, concentrations[:, 0], 'ro', alpha=0.6, label='A (data)')
    axes[0, 0].plot(t_rxn, concentrations[:, 1], 'go', alpha=0.6, label='B (data)')
    axes[0, 0].plot(t_rxn, concentrations[:, 2], 'bo', alpha=0.6, label='C (data)')
    axes[0, 0].plot(t_test, conc_pred[:, 0], 'r-', linewidth=2, label='A (PINN)')
    axes[0, 0].plot(t_test, conc_pred[:, 1], 'g-', linewidth=2, label='B (PINN)')
    axes[0, 0].plot(t_test, conc_pred[:, 2], 'b-', linewidth=2, label='C (PINN)')
    axes[0, 0].set_xlabel('Time')
    axes[0, 0].set_ylabel('Concentration')
    axes[0, 0].set_title('Reaction Kinetics: A + B → C')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Rate constant discovery
    axes[0, 1].plot(k_rxn_hist, 'orange', linewidth=2, label='Learned k1')
    axes[0, 1].axhline(y=kinetics_exp.true_rate_constants[0], color='red', 
                      linestyle='--', linewidth=2, label='True k1')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Rate Constant')
    axes[0, 1].set_title('Rate Constant Discovery')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Mass conservation check
    total_mass_data = np.sum(concentrations, axis=1)
    total_mass_pred = np.sum(conc_pred, axis=1)
    
    axes[1, 0].plot(t_rxn, total_mass_data, 'ko', alpha=0.6, label='Data')
    axes[1, 0].plot(t_test, total_mass_pred, 'r-', linewidth=2, label='PINN')
    axes[1, 0].set_xlabel('Time')
    axes[1, 0].set_ylabel('Total Mass')
    axes[1, 0].set_title('Mass Conservation Check')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Training loss
    axes[1, 1].semilogy(losses_rxn, 'blue', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss')
    axes[1, 1].set_title('Training Loss')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print discovered parameters
    print("\n" + "="*60)
    print("DISCOVERY RESULTS SUMMARY")
    print("="*60)
    
    final_mass = mass_hist[-1]
    final_k = k_hist[-1]
    final_k_rxn = k_rxn_hist[-1]
    
    print(f"\nNewton's Law Discovery:")
    print(f"  True mass: {newton_exp.true_mass:.3f} | Discovered: {final_mass:.3f} | Error: {abs(final_mass - newton_exp.true_mass)/newton_exp.true_mass*100:.1f}%")
    print(f"  True k: {newton_exp.true_force_params[0]:.3f} | Discovered: {final_k:.3f} | Error: {abs(final_k - newton_exp.true_force_params[0])/newton_exp.true_force_params[0]*100:.1f}%")
    
    print(f"\nReaction Kinetics Discovery:")
    print(f"  True k1: {kinetics_exp.true_rate_constants[0]:.3f} | Discovered: {final_k_rxn:.3f} | Error: {abs(final_k_rxn - kinetics_exp.true_rate_constants[0])/kinetics_exp.true_rate_constants[0]*100:.1f}%")
    
    print(f"\nEnergy Conservation:")
    print(f"  Final energy variance: {energy_violations[-1]:.6f}")
    print(f"  Conservation quality: {'EXCELLENT' if energy_violations[-1] < 1e-4 else 'GOOD' if energy_violations[-1] < 1e-3 else 'NEEDS IMPROVEMENT'}")


In [ ]:

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("PHYSICS-INFORMED NEURAL NETWORKS FOR SCIENTIFIC DISCOVERY")
    print("=" * 70)
    print("This lab demonstrates how PINNs can:")
    print("1. Discover governing equations from experimental data")
    print("2. Verify known physical laws")
    print("3. Quantify model uncertainty and violations")
    print("=" * 70)
    
    # Run all discovery experiments
    visualize_discovery_results()
    
    print("\n" + "="*70)
    print("EXPERIMENTAL EXTENSIONS FOR STUDENTS:")
    print("="*70)
    print("1. Try different noise levels and see how it affects discovery")
    print("2. Use incomplete data (missing some variables) for discovery")
    print("3. Add unknown physics (e.g., friction) and see if PINN finds it")
    print("4. Test with nonlinear systems (double pendulum, Lorenz system)")
    print("5. Discover partial differential equations from field data")
    print("6. Use multi-fidelity data (high/low resolution measurements)")
    print("7. Implement uncertainty quantification for discovered parameters")
    print("="*70)

In [ ]:
"""
Robust Physics-Informed Neural Networks for Wave Functions
=========================================================

Adaptive loss weighting strategies, multiple training phases,
better eigenvalue initialization, enhanced architectures,
robust boundary condition enforcement.
"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scipy.special as sp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class RobustComplexPINN(nn.Module):
    """Enhanced PINN with residual connections"""
    
    def __init__(self, input_dim=1, hidden_layers=6, hidden_units=128):
        super(RobustComplexPINN, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_units = hidden_units
        
        self.input_layer = nn.Linear(input_dim, hidden_units)
        
        self.hidden_layers = nn.ModuleList()
        for i in range(hidden_layers):
            self.hidden_layers.append(nn.Linear(hidden_units, hidden_units))
        
        self.real_output = nn.Linear(hidden_units, 1)
        self.imag_output = nn.Linear(hidden_units, 1)
        
        self.activation = nn.Tanh()
        
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight, gain=1.0)
            nn.init.zeros_(module.bias)
    
    def forward(self, x):
        if self.input_dim == 1:
            x_norm = 2.0 * x - 1.0  # Map [0,1] to [-1,1]
        else:
            x_norm = x
        
        h = self.activation(self.input_layer(x_norm))
        
        for i, layer in enumerate(self.hidden_layers):
            h_new = self.activation(layer(h))
            if i % 2 == 1:  # Add residual every 2 layers
                h = h + h_new
            else:
                h = h_new
        
        psi_real = self.real_output(h)
        psi_imag = self.imag_output(h)
        
        return psi_real, psi_imag

class AdaptiveParticleInBox:
    """Adaptive training strategy for particle in box"""
    
    def __init__(self, L=1.0, hbar=1.0, m=1.0):
        self.L = L
        self.hbar = hbar
        self.m = m
    
    def analytical_solution(self, x, n=1):
        return np.sqrt(2/self.L) * np.sin(n * np.pi * x / self.L)
    
    def analytical_energy(self, n=1):
        return (n**2 * np.pi**2 * self.hbar**2) / (2 * self.m * self.L**2)
    
    def create_training_data(self, n_collocation=2000, n_boundary=100):
        
        x_interior = torch.rand(n_collocation//2, 1) * self.L
        
        # Extra points near boundaries
        x_boundary_region = torch.cat([
            torch.rand(n_collocation//4, 1) * 0.1 * self.L,
            0.9 * self.L + torch.rand(n_collocation//4, 1) * 0.1 * self.L
        ])
        
        x_train = torch.cat([x_interior, x_boundary_region]).to(device)
        x_boundary = torch.tensor([[0.0], [self.L]], device=device)
        
        return x_train, x_boundary
    
    def train_eigenstate_adaptive(self, n_state=1, total_epochs=12000, lr_start=1e-3):
        """Multi-phase adaptive training strategy"""
        
        print(f"Training Particle in Box (n={n_state}) with adaptive strategy...")
        
        model = RobustComplexPINN(input_dim=1, hidden_layers=8, hidden_units=150).to(device)
        
        analytical_E = self.analytical_energy(n_state)
        eigenvalue = nn.Parameter(torch.tensor([analytical_E * 0.7], device=device))
        
        x_train, x_boundary = self.create_training_data()
        
        phases = [
            {"epochs": 3000, "lr_model": lr_start, "lr_eigen": lr_start * 2, 
             "weights": {"pde": 100, "bc": 1000, "norm": 50, "ortho": 10}},
            {"epochs": 4000, "lr_model": lr_start * 0.5, "lr_eigen": lr_start * 3, 
             "weights": {"pde": 1000, "bc": 800, "norm": 100, "ortho": 20}},
            {"epochs": 3000, "lr_model": lr_start * 0.2, "lr_eigen": lr_start * 2, 
             "weights": {"pde": 2000, "bc": 500, "norm": 200, "ortho": 50}},
            {"epochs": 2000, "lr_model": lr_start * 0.1, "lr_eigen": lr_start * 1, 
             "weights": {"pde": 5000, "bc": 200, "norm": 500, "ortho": 100}}
        ]
        
        all_losses = []
        all_eigenvalues = []
        best_eigenvalue = analytical_E
        best_loss = float('inf')
        
        current_epoch = 0
        
        for phase_idx, phase in enumerate(phases):
            print(f"\n--- Phase {phase_idx + 1}: Focusing on {'boundary' if phase_idx == 0 else 'PDE' if phase_idx < 2 else 'fine-tuning'} ---")
            
            model_optimizer = torch.optim.Adam(model.parameters(), lr=phase["lr_model"])
            eigenvalue_optimizer = torch.optim.Adam([eigenvalue], lr=phase["lr_eigen"])
            
            model_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                model_optimizer, T_max=phase["epochs"], eta_min=phase["lr_model"] * 0.1)
            eigen_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                eigenvalue_optimizer, T_max=phase["epochs"], eta_min=phase["lr_eigen"] * 0.1)
            
            phase_losses = []
            phase_eigenvalues = []
            
            for epoch in range(phase["epochs"]):
                model_optimizer.zero_grad()
                eigenvalue_optimizer.zero_grad()
                
                x_train.requires_grad_(True)
                psi_real, psi_imag = model(x_train)
                psi = psi_real
                
                psi_x = torch.autograd.grad(psi.sum(), x_train, create_graph=True)[0]
                psi_xx = torch.autograd.grad(psi_x.sum(), x_train, create_graph=True)[0]
                
                # Schrödinger equation
                kinetic_coeff = (2 * self.m) / (self.hbar**2)
                pde_residual = psi_xx + kinetic_coeff * eigenvalue * psi
                loss_pde = torch.mean(pde_residual**2)
                
                # Boundary conditions
                psi_boundary, _ = model(x_boundary)
                loss_bc = torch.mean(psi_boundary**2)
                
                # Normalization
                dx = self.L / len(x_train)
                norm_squared = torch.sum(psi**2) * dx
                loss_norm = (norm_squared - 1.0)**2
                
                # Orthogonality for excited states
                loss_ortho = 0.0
                if n_state > 1:
                    # Node enforcement
                    nodes = [k * self.L / n_state for k in range(1, n_state)]
                    if nodes:
                        x_nodes = torch.tensor([[node] for node in nodes], device=device)
                        psi_nodes, _ = model(x_nodes)
                        loss_ortho = torch.mean(psi_nodes**2)
                    
                    # Symmetry enforcement
                    if n_state % 2 == 0:  # Even states
                        x_mid = torch.linspace(0, self.L/2, 100).reshape(-1, 1).to(device)
                        x_mirror = self.L - x_mid
                        psi_mid, _ = model(x_mid)
                        psi_mirror, _ = model(x_mirror)
                        loss_ortho += torch.mean((psi_mid - psi_mirror)**2)
                    else:  # Odd states
                        x_mid = torch.linspace(0, self.L/2, 100).reshape(-1, 1).to(device)
                        x_mirror = self.L - x_mid
                        psi_mid, _ = model(x_mid)
                        psi_mirror, _ = model(x_mirror)
                        loss_ortho += torch.mean((psi_mid + psi_mirror)**2)
                
                weights = phase["weights"]
                total_loss = (weights["pde"] * loss_pde + 
                             weights["bc"] * loss_bc + 
                             weights["norm"] * loss_norm + 
                             weights["ortho"] * loss_ortho)
                
                total_loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                torch.nn.utils.clip_grad_norm_([eigenvalue], max_norm=5.0)
                
                model_optimizer.step()
                eigenvalue_optimizer.step()
                model_scheduler.step()
                eigen_scheduler.step()
                
                if total_loss.item() < best_loss:
                    best_loss = total_loss.item()
                    best_eigenvalue = eigenvalue.item()
                
                phase_losses.append(total_loss.item())
                phase_eigenvalues.append(eigenvalue.item())
                
                if epoch % 500 == 0:
                    error_percent = abs(eigenvalue.item() - analytical_E) / analytical_E * 100
                    print(f"  Epoch {current_epoch + epoch:5d} | Loss: {total_loss.item():.2e} | "
                          f"E: {eigenvalue.item():.4f} (Target: {analytical_E:.4f}, Error: {error_percent:.1f}%)")
            
            all_losses.extend(phase_losses)
            all_eigenvalues.extend(phase_eigenvalues)
            current_epoch += phase["epochs"]
            
            final_error = abs(eigenvalue.item() - analytical_E) / analytical_E * 100
            print(f"  Phase {phase_idx + 1} complete: Final error = {final_error:.2f}%")
        
        final_error = abs(best_eigenvalue - analytical_E) / analytical_E * 100
        print(f"\nFinal Result: E_predicted={best_eigenvalue:.4f}, E_analytical={analytical_E:.4f}")
        print(f"Final Error: {final_error:.2f}%")
        
        return model, torch.tensor([best_eigenvalue], device=device), all_losses, all_eigenvalues

class SimpleQuantumHarmonicOscillator:
    
    def __init__(self, hbar=1.0, m=1.0, omega=1.0):
        self.hbar = hbar
        self.m = m
        self.omega = omega
    
    def analytical_energy(self, n=0):
        return self.hbar * self.omega * (n + 0.5)
    
    def analytical_solution(self, x, n=0):
        alpha = np.sqrt(self.m * self.omega / self.hbar)
        xi = alpha * x
        norm = (alpha / np.sqrt(np.pi))**(1/2) / np.sqrt(2**n * np.math.factorial(n))
        H_n = sp.hermite(n)(xi)
        return norm * H_n * np.exp(-xi**2 / 2)
    
    def train_ground_state(self, epochs=6000, lr=1e-3):
        print("Training Quantum Harmonic Oscillator (ground state)...")
        
        model = RobustComplexPINN(input_dim=1, hidden_layers=6, hidden_units=120).to(device)
        
        analytical_E = self.analytical_energy(0)
        eigenvalue = nn.Parameter(torch.tensor([analytical_E * 0.8], device=device))
        
        model_optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        eigenvalue_optimizer = torch.optim.Adam([eigenvalue], lr=lr * 3.0)
        
        x_train = torch.linspace(-4, 4, 1000).reshape(-1, 1).to(device)
        x_boundary = torch.tensor([[-4.0], [4.0]], device=device)
        
        losses = []
        eigenvalue_history = []
        
        for epoch in range(epochs):
            model_optimizer.zero_grad()
            eigenvalue_optimizer.zero_grad()
            
            x_train.requires_grad_(True)
            psi_real, psi_imag = model(x_train)
            psi = psi_real
            
            psi_x = torch.autograd.grad(psi.sum(), x_train, create_graph=True)[0]
            psi_xx = torch.autograd.grad(psi_x.sum(), x_train, create_graph=True)[0]
            
            # Harmonic oscillator potential
            V = 0.5 * self.m * self.omega**2 * x_train**2
            
            # Schrödinger equation: -ℏ²/(2m) ψ'' + V ψ = E ψ
            kinetic_term = -(self.hbar**2) / (2 * self.m) * psi_xx
            potential_term = V * psi
            energy_term = eigenvalue * psi
            
            schrodinger_residual = kinetic_term + potential_term - energy_term
            loss_pde = torch.mean(schrodinger_residual**2)
            
            # Boundary conditions (decay at infinity)
            psi_boundary, _ = model(x_boundary)
            loss_bc = torch.mean(psi_boundary**2)
            
            # Normalization
            dx = 8.0 / len(x_train)
            norm_squared = torch.sum(psi**2) * dx
            loss_norm = (norm_squared - 1.0)**2
            
            # Ground state should be even (symmetric)
            x_neg = -x_train
            psi_neg, _ = model(x_neg)
            loss_symmetry = torch.mean((psi - psi_neg)**2)
            
            total_loss = 1000.0 * loss_pde + 100.0 * loss_bc + 50.0 * loss_norm + 20.0 * loss_symmetry
            
            total_loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_([eigenvalue], max_norm=2.0)
            
            model_optimizer.step()
            eigenvalue_optimizer.step()
            
            losses.append(total_loss.item())
            eigenvalue_history.append(eigenvalue.item())
            
            if epoch % 1000 == 0:
                error_percent = abs(eigenvalue.item() - analytical_E) / analytical_E * 100
                print(f"Epoch {epoch:4d} | Loss: {total_loss.item():.2e} | "
                      f"E: {eigenvalue.item():.4f} (True: {analytical_E:.4f}, Error: {error_percent:.1f}%)")
        
        return model, eigenvalue, losses, eigenvalue_history

def run_robust_analysis():
    
    print("\n" + "="*70)
    print("ROBUST WAVE FUNCTION ANALYSIS")
    print("="*70)
    
    print("\n1. ADAPTIVE PARTICLE IN BOX TRAINING")
    print("-" * 50)
    
    pib = AdaptiveParticleInBox(L=1.0)
    results_pib = {}
    
    for n in [1, 2, 3]:
        print(f"\n=== Training quantum state n = {n} ===")
        model, energy, losses, E_hist = pib.train_eigenstate_adaptive(
            n_state=n, total_epochs=10000, lr_start=1e-3)
        
        results_pib[n] = {
            'model': model,
            'energy': energy.item(),
            'losses': losses,
            'E_hist': E_hist
        }
    
    print("\n\n2. QUANTUM HARMONIC OSCILLATOR")
    print("-" * 50)
    
    qho = SimpleQuantumHarmonicOscillator()
    model_qho, energy_qho, losses_qho, E_hist_qho = qho.train_ground_state(epochs=5000)
    
    create_robust_visualizations(results_pib, model_qho, energy_qho, losses_qho, E_hist_qho, pib, qho)
    print_robust_results(results_pib, energy_qho, pib, qho)
    
    return results_pib, model_qho, energy_qho

def create_robust_visualizations(results_pib, model_qho, energy_qho, losses_qho, E_hist_qho, pib, qho):
    
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Energy accuracy comparison
    ax1 = plt.subplot(2, 4, 1)
    n_values = [1, 2, 3]
    predicted_energies = [results_pib[n]['energy'] for n in n_values]
    analytical_energies = [pib.analytical_energy(n) for n in n_values]
    errors = [abs(p - a)/a * 100 for p, a in zip(predicted_energies, analytical_energies)]
    
    x_pos = np.arange(len(n_values))
    width = 0.35
    
    bars1 = ax1.bar(x_pos - width/2, predicted_energies, width, label='PINN', alpha=0.8, color='blue')
    bars2 = ax1.bar(x_pos + width/2, analytical_energies, width, label='Analytical', alpha=0.8, color='red')
    
    for i, (pred, anal, err) in enumerate(zip(predicted_energies, analytical_energies, errors)):
        ax1.text(i, max(pred, anal) + 2, f'{err:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax1.set_xlabel('Quantum Number n')
    ax1.set_ylabel('Energy')
    ax1.set_title('Energy Prediction Accuracy')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(n_values)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Wave function comparison
    ax2 = plt.subplot(2, 4, 2)
    x_test = torch.linspace(0, 1, 300).reshape(-1, 1).to(device)
    
    for n in [1, 2, 3]:
        with torch.no_grad():
            psi_pred, _ = results_pib[n]['model'](x_test)
            psi_pred = psi_pred.cpu().numpy().flatten()
        
        x_np = x_test.cpu().numpy().flatten()
        psi_analytical = pib.analytical_solution(x_np, n)
        
        norm = np.sqrt(np.trapz(psi_pred**2, x_np))
        if norm > 0:
            psi_pred = psi_pred / norm
        
        # Align sign
        if np.trapz(psi_pred * psi_analytical, x_np) < 0:
            psi_pred = -psi_pred
        
        offset = (n - 1) * 2.5
        ax2.plot(x_np, psi_pred + offset, 'b-', linewidth=2, alpha=0.8)
        ax2.plot(x_np, psi_analytical + offset, 'r--', linewidth=2, alpha=0.7)
        ax2.text(0.05, offset + 1, f'n={n}', fontweight='bold')
    
    ax2.set_xlabel('Position x')
    ax2.set_ylabel('ψ(x) + offset')
    ax2.set_title('Wave Functions: Blue=PINN, Red=Analytical')
    ax2.grid(True, alpha=0.3)
    
    # 3. Training convergence
    ax3 = plt.subplot(2, 4, 3)
    colors = ['blue', 'green', 'red']
    for i, n in enumerate([1, 2, 3]):
        E_hist = results_pib[n]['E_hist']
        analytical_E = pib.analytical_energy(n)
        
        epochs = range(len(E_hist))
        ax3.plot(epochs, E_hist, color=colors[i], linewidth=2, label=f'n={n}')
        ax3.axhline(y=analytical_E, color=colors[i], linestyle='--', alpha=0.7)
    
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Energy')
    ax3.set_title('Energy Convergence')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Loss evolution
    ax4 = plt.subplot(2, 4, 4)
    for i, n in enumerate([1, 2, 3]):
        losses = results_pib[n]['losses']
        epochs = range(len(losses))
        ax4.semilogy(epochs, losses, color=colors[i], linewidth=2, label=f'n={n}')
    
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Loss')
    ax4.set_title('Training Loss Evolution')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Scaling verification
    ax5 = plt.subplot(2, 4, 5)
    n_vals = [1, 2, 3]
    predicted_ratios = [results_pib[n]['energy'] / results_pib[1]['energy'] for n in n_vals]
    theoretical_ratios = [n**2 for n in n_vals]
    
    ax5.plot(n_vals, predicted_ratios, 'bo-', linewidth=2, markersize=8, label='PINN')
    ax5.plot(n_vals, theoretical_ratios, 'r--', linewidth=2, markersize=8, label='Theory (n²)')
    
    for i, (n, pred, theo) in enumerate(zip(n_vals, predicted_ratios, theoretical_ratios)):
        ax5.text(n, pred + 0.5, f'{pred:.1f}', ha='center', va='bottom', fontweight='bold')
    
    ax5.set_xlabel('Quantum Number n')
    ax5.set_ylabel('Energy Ratio En/E1')
    ax5.set_title('n² Scaling Verification')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Harmonic oscillator comparison
    ax6 = plt.subplot(2, 4, 6)
    x_qho = torch.linspace(-4, 4, 200).reshape(-1, 1).to(device)
    
    with torch.no_grad():
        psi_qho_pred, _ = model_qho(x_qho)
        psi_qho_pred = psi_qho_pred.cpu().numpy().flatten()
    
    x_qho_np = x_qho.cpu().numpy().flatten()
    psi_qho_analytical = qho.analytical_solution(x_qho_np, 0)
    
    norm_qho = np.sqrt(np.trapz(psi_qho_pred**2, x_qho_np))
    if norm_qho > 0:
        psi_qho_pred = psi_qho_pred / norm_qho
    
    # Align sign
    if np.trapz(psi_qho_pred * psi_qho_analytical, x_qho_np) < 0:
        psi_qho_pred = -psi_qho_pred
    
    ax6.plot(x_qho_np, psi_qho_pred, 'b-', linewidth=2, label='PINN')
    ax6.plot(x_qho_np, psi_qho_analytical, 'r--', linewidth=2, label='Analytical')
    ax6.set_xlabel('Position x')
    ax6.set_ylabel('ψ₀(x)')
    ax6.set_title('Harmonic Oscillator Ground State')
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # 7. Error analysis
    ax7 = plt.subplot(2, 4, 7)
    systems = ['PIB n=1', 'PIB n=2', 'PIB n=3', 'QHO n=0']
    errors_all = []
    
    for n in [1, 2, 3]:
        analytical_E = pib.analytical_energy(n)
        predicted_E = results_pib[n]['energy']
        error = abs(predicted_E - analytical_E) / analytical_E * 100
        errors_all.append(error)
    
    qho_analytical = qho.analytical_energy(0)
    qho_predicted = energy_qho.item()
    qho_error = abs(qho_predicted - qho_analytical) / qho_analytical * 100
    errors_all.append(qho_error)
    
    colors_bar = ['lightblue', 'lightgreen', 'lightcoral', 'gold']
    bars = ax7.bar(systems, errors_all, color=colors_bar, alpha=0.8)
    
    for bar, error in zip(bars, errors_all):
        ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{error:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax7.set_ylabel('Error (%)')
    ax7.set_title('Prediction Accuracy Summary')
    ax7.tick_params(axis='x', rotation=45)
    ax7.grid(True, alpha=0.3)
    
    # 8. Physics validation
    ax8 = plt.subplot(2, 4, 8)
    
    scaling_errors = []
    for n in [2, 3]:
        expected_ratio = n**2
        actual_ratio = results_pib[n]['energy'] / results_pib[1]['energy']
        scaling_error = abs(actual_ratio - expected_ratio) / expected_ratio * 100
        scaling_errors.append(scaling_error)
    
    ratios = ['E₂/E₁', 'E₃/E₁']
    bars_scale = ax8.bar(ratios, scaling_errors, color=['orange', 'purple'], alpha=0.8)
    
    for bar, error in zip(bars_scale, scaling_errors):
        ax8.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{error:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax8.set_ylabel('Scaling Error (%)')
    ax8.set_title('n² Scaling Accuracy')
    ax8.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def print_robust_results(results_pib, energy_qho, pib, qho):
    
    print("\n" + "="*70)
    print("ROBUST TRAINING RESULTS")
    print("="*70)
    
    print(f"\nPARTICLE IN BOX ANALYSIS:")
    print(f"{'n':>3} | {'Predicted':>10} | {'Analytical':>10} | {'Error (%)':>10} | {'Status':>15}")
    print("-" * 65)
    
    total_error = 0
    for n in [1, 2, 3]:
        analytical_E = pib.analytical_energy(n)
        predicted_E = results_pib[n]['energy']
        error_percent = abs(predicted_E - analytical_E) / analytical_E * 100
        total_error += error_percent
        
        if error_percent < 3.0:
            status = "EXCELLENT"
        elif error_percent < 8.0:
            status = "GOOD"
        elif error_percent < 20.0:
            status = "FAIR"
        else:
            status = "NEEDS WORK"
        
        print(f"{n:>3} | {predicted_E:>10.4f} | {analytical_E:>10.4f} | {error_percent:>9.2f}% | {status:>15}")
    
    avg_error = total_error / 3
    print(f"\nAverage Error: {avg_error:.2f}%")
    
    print(f"\nHARMONIC OSCILLATOR ANALYSIS:")
    qho_analytical = qho.analytical_energy(0)
    qho_predicted = energy_qho.item()
    qho_error = abs(qho_predicted - qho_analytical) / qho_analytical * 100
    
    qho_status = "EXCELLENT" if qho_error < 3.0 else ("GOOD" if qho_error < 8.0 else "FAIR")
    
    print(f"Ground State (n=0):")
    print(f"  Predicted Energy: {qho_predicted:.4f}")
    print(f"  Analytical Energy: {qho_analytical:.4f}")
    print(f"  Error: {qho_error:.2f}% - {qho_status}")
    
    print(f"\nPHYSICS VALIDATION:")
    print("Quantum number scaling (E_n ∝ n²):")
    print(f"{'Ratio':>15} | {'Predicted':>12} | {'Theory':>10} | {'Error (%)':>10}")
    print("-" * 60)
    
    for n in [2, 3]:
        ratio_pred = results_pib[n]['energy'] / results_pib[1]['energy']
        ratio_theory = n**2
        ratio_error = abs(ratio_pred - ratio_theory) / ratio_theory * 100
        
        print(f"E_{n}/E_1:          | {ratio_pred:>11.2f} | {ratio_theory:>9.0f} | {ratio_error:>9.2f}%")
    
    print(f"\nOVERALL ASSESSMENT:")
    if avg_error < 5.0:
        print("OUTSTANDING: Physics-informed neural networks successfully learned quantum mechanics.")
        print("All energy levels predicted with high accuracy.")
    elif avg_error < 15.0:
        print("SUCCESS: Good agreement with quantum mechanical theory.")
        print("Minor discrepancies likely due to training convergence.")
    elif avg_error < 30.0:
        print("PARTIAL SUCCESS: Basic quantum behavior captured.")
        print("Some training improvements needed for higher accuracy.")
    else:
        print("NEEDS IMPROVEMENT: Significant deviations from expected results.")
        print("Consider longer training or architecture modifications.")
    
    print(f"\nTRAINING INSIGHTS:")
    print("- Multi-phase adaptive training strategy used")
    print("- Separate learning rates for network vs eigenvalue parameters")
    print("- Enhanced boundary condition enforcement")
    print("- Residual connections for better gradient flow")
    print("- Gradient clipping for training stability")


if __name__ == "__main__":
    print("ROBUST PHYSICS-INFORMED NEURAL NETWORKS FOR WAVE FUNCTIONS")
    print("=" * 70)
    print("Advanced Training Strategies for Quantum Mechanics")
    print("=" * 70)
    
    results_pib, model_qho, energy_qho = run_robust_analysis()


    avg_pib_error = sum(abs(results_pib[n]['energy'] - AdaptiveParticleInBox().analytical_energy(n)) / 
                       AdaptiveParticleInBox().analytical_energy(n) * 100 for n in [1,2,3]) / 3
    
    print(f"\nFinal average error: {avg_pib_error:.1f}%")